# Análisis de Series Temporales — TP1
## Movilidad Urbana en Buenos Aires: Subte, Peajes y Precipitaciones

**Universidad Austral · Maestría en Gestión de Ciencia de Datos**  

**Integrantes:**
- Alvaro, Giuliana
- Chalup, Sarah
- Fontán, Mariana
- Franco, Agustina

**Fecha de entrega:** 16 de julio de 2025

---
## 5. Setup: Librerías e Imports

In [ ]:
# ── Manipulación de datos ──────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Visualización ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ── Series temporales ─────────────────────────────────────────────────────
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import jarque_bera

# ── Métricas ──────────────────────────────────────────────────────────────
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

# ── Descarga de datos ─────────────────────────────────────────────────────
import requests
import zipfile
import io
import os
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# ── Configuración de plots ─────────────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize': (14, 4),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

# ── Directorio de datos ────────────────────────────────────────────────────
DATA_DIR = Path('datos_tp1')
DATA_DIR.mkdir(exist_ok=True)

print('Setup completo ✓')

Setup completo ✓


---
## ⚡ Carga Rápida

Si ya descargaste los datos al menos una vez, **ejecutá solo esta celda** y saltá directo a la **Sección 7 — EDA**.  
No hace falta volver a descargar nada.

In [13]:
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('datos_tp1')

NECESARIOS = ['subte_mensual.csv', 'lluvia_mensual.csv']   # agregar 'peajes_mensual.csv' cuando esté
faltantes  = [f for f in NECESARIOS if not (DATA_DIR / f).exists()]

if faltantes:
    print(f'⚠️  Faltan archivos: {faltantes}')
    print('   Corré primero la Sección 6 (Descarga y Procesamiento de Datos).')
else:
    subte_mensual  = pd.read_csv(DATA_DIR / 'subte_mensual.csv',
                                  index_col=0, parse_dates=True).squeeze()
    lluvia_mensual = pd.read_csv(DATA_DIR / 'lluvia_mensual.csv',
                                  index_col=0, parse_dates=True).squeeze()
    # peajes_mensual = pd.read_csv(DATA_DIR / 'peajes_mensual.csv',
    #                               index_col=0, parse_dates=True).squeeze()

    START, END     = '2013-06-01', '2025-12-01'
    TRAIN_END      = '2023-12-01'
    TEST_START     = '2024-01-01'

    df = pd.DataFrame({
        'pasajeros_subte': subte_mensual,
        # 'vehiculos_peajes': peajes_mensual,
        'lluvia_mm':       lluvia_mensual,
    }).loc[START:END]
    df['dummy_covid'] = 0
    df.loc['2020-03-01':'2021-12-01', 'dummy_covid'] = 1

    train = df.loc[:TRAIN_END]
    test  = df.loc[TEST_START:]

    print('✅ Datos cargados correctamente. Podés saltar a la Sección 7.')
    print(f'   Período  : {df.index[0].date()} -> {df.index[-1].date()} ({len(df)} obs)')
    print(f'   Train    : {train.index[0].date()} -> {train.index[-1].date()} ({len(train)} obs)')
    print(f'   Test     : {test.index[0].date()} -> {test.index[-1].date()} ({len(test)} obs)')
    print(f'   COVID    : {df["dummy_covid"].sum()} meses marcados')
    display(df.tail())


✅ Datos cargados correctamente. Podés saltar a la Sección 7.
   Período  : 2013-06-01 -> 2025-12-01 (151 obs)
   Train    : 2013-06-01 -> 2023-12-01 (127 obs)
   Test     : 2024-01-01 -> 2025-12-01 (24 obs)
   COVID    : 22 meses marcados


,pasajeros_subte,lluvia_mm,dummy_covid
fecha,,,
2025-08-01,NaN,124.0,0
2025-09-01,NaN,94.7,0
2025-10-01,NaN,94.8,0
2025-11-01,NaN,89.5,0
2025-12-01,NaN,47.5,0


---
## 6. Descarga y Procesamiento de Datos

### 6.1 Precipitaciones — Open-Meteo API

In [12]:
def descargar_precipitaciones(start='2013-06-01', end='2025-12-31'):
    """Descarga datos diarios de precipitación de Open-Meteo y agrega a mensual."""
    url = (
        f"https://archive-api.open-meteo.com/v1/archive"
        f"?latitude=-34.60&longitude=-58.37"
        f"&start_date={start}&end_date={end}"
        f"&daily=precipitation_sum"
        f"&timezone=America%2FArgentina%2FBuenos_Aires"
        f"&format=csv"
    )
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    
    # El CSV de Open-Meteo tiene 3 líneas de encabezado
    df = pd.read_csv(io.StringIO(resp.text), skiprows=3, parse_dates=['time'])
    df = df.rename(columns={'time': 'fecha', 'precipitation_sum (mm)': 'lluvia_mm'})
    df = df.set_index('fecha')
    
    # Agregar a mensual
    lluvia_mensual = df['lluvia_mm'].resample('MS').sum()
    lluvia_mensual.name = 'lluvia_mm'
    return lluvia_mensual

lluvia = descargar_precipitaciones()
lluvia.to_csv(DATA_DIR / 'lluvia_mensual.csv')
print(f'Precipitaciones: {len(lluvia)} observaciones mensuales')
print(f'Período: {lluvia.index[0].date()} → {lluvia.index[-1].date()}')
lluvia.head()

Precipitaciones: 151 observaciones mensuales
Período: 2013-06-01 → 2025-12-01


fecha
2013-06-01      8.6
2013-07-01     78.2
2013-08-01      6.8
2013-09-01    102.4
2013-10-01     27.5
Freq: MS, Name: lluvia_mm, dtype: float64

### 6.2 Subte — Molinetes GCBA

> Los archivos CSV/ZIP se descargan desde el portal de datos abiertos del GCBA.  
> Por el tamaño de los archivos, se procesan año por año y se consolidan en un único CSV mensual.

In [ ]:
# ── URLs verificadas (abril 2026) ─────────────────────────────────────────
SUBTE_URLS = {
    2013: 'https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-viajes-molinetes/molinetes-2013-junio-diciembre.zip',
    2014: 'https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-viajes-molinetes/molinetes-2014.zip',
    2015: 'https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-viajes-molinetes/molinetes-2015.zip',
    2016: 'https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-viajes-molinetes/molinetes-2016.zip',
    2017: 'https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-viajes-molinetes/molinetes-2017.zip',
    2018: 'https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-viajes-molinetes/molinetes-2018.zip',
    2019: 'https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-viajes-molinetes/molinetes-2019.zip',
    2020: 'https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-viajes-molinetes/molinetes-2020.zip',
    2021: 'https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-viajes-molinetes/molinetes-2021.zip',
    2022: 'https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-viajes-molinetes/molinetes-2022.zip',
    2023: 'https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-viajes-molinetes/molinetes-2023.zip',
    2024: 'https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-viajes-molinetes/molinetes-2024.zip',
    2025: 'https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-viajes-molinetes/molinetes-2025.zip',
}

def limpiar_raw(raw_bytes):
    """
    Quita BOM y comillas externas de cada línea.
    El portal GCBA envuelve cada fila entre comillas dobles en 2022-2025:
      \"1/1/2022;08:00:00;...;1\"  ->  1/1/2022;08:00:00;...;1
    Devuelve (bytes_limpios, encoding).
    """
    if raw_bytes.startswith(b'\xef\xbb\xbf'):
        raw_bytes = raw_bytes[3:]
        enc = 'utf-8'
    else:
        enc = 'latin1'

    texto = raw_bytes.decode(enc)
    lineas = texto.splitlines()

    # Si la primera línea empieza y termina con comillas, limpiar todas
    primera = lineas[0].strip() if lineas else ''
    if primera.startswith('"') and primera.endswith('"'):
        lineas = [l.strip().strip('"') for l in lineas if l.strip()]

    limpio = '\n'.join(lineas).encode(enc)
    return limpio, enc

def procesar_subte_anio(url, anio):
    """
    Descarga un ZIP anual de molinetes y devuelve [fecha, pax].
    Maneja todos los formatos del portal GCBA (2013-2025).
    """
    cache_path = DATA_DIR / f'subte_{anio}.csv'
    if cache_path.exists():
        print(f'  {anio}: cache local OK')
        return pd.read_csv(cache_path, parse_dates=['fecha'])

    print(f'  {anio}: descargando...', end=' ', flush=True)
    try:
        resp = requests.get(url, timeout=300)
        resp.raise_for_status()

        z = zipfile.ZipFile(io.BytesIO(resp.content))
        csv_name = [n for n in z.namelist() if n.lower().endswith('.csv')][0]
        raw = z.open(csv_name).read()

        # Limpiar BOM y comillas externas por línea
        raw_limpio, enc = limpiar_raw(raw)

        # Detectar separador desde la primera línea ya limpia
        primera = raw_limpio.split(b'\n')[0].decode(enc)
        sep = ';' if ';' in primera else ','

        df_raw = pd.read_csv(io.BytesIO(raw_limpio), sep=sep, encoding=enc, low_memory=False)

        # Normalizar columnas
        df_raw.columns = df_raw.columns.str.strip().str.lower()

        if 'fecha' not in df_raw.columns:
            raise ValueError(f'Sin col fecha. Columnas: {df_raw.columns.tolist()}')

        pax_col = next(
            (c for c in df_raw.columns if c == 'pax_total'),
            next((c for c in df_raw.columns if c.startswith('pax_')), None)
        )
        if pax_col is None:
            raise ValueError(f'Sin col pax. Columnas: {df_raw.columns.tolist()}')

        df_raw['pax']   = pd.to_numeric(df_raw[pax_col], errors='coerce')
        df_raw['fecha'] = pd.to_datetime(df_raw['fecha'], dayfirst=True, errors='coerce')

        out = df_raw[['fecha', 'pax']].dropna()
        out.to_csv(cache_path, index=False)
        print(f'OK  ({len(out):,} registros | sep={repr(sep)} | enc={enc})')
        return out

    except Exception as e:
        print(f'ERROR: {e}')
        return pd.DataFrame(columns=['fecha', 'pax'])

# ── Descargar / leer desde caché ──────────────────────────────────────────
print('Procesando subte por año...')
frames = [procesar_subte_anio(url, anio) for anio, url in SUBTE_URLS.items()]
subte_raw = pd.concat([f for f in frames if not f.empty], ignore_index=True)

if subte_raw.empty:
    print('\n⚠️  No se cargó ningún año.')
else:
    subte_raw['fecha'] = pd.to_datetime(subte_raw['fecha'])
    subte_mensual = subte_raw.set_index('fecha')['pax'].resample('MS').sum()
    subte_mensual.name = 'pasajeros'
    subte_mensual.to_csv(DATA_DIR / 'subte_mensual.csv')
    print(f'\n✅ subte_mensual.csv guardado — {len(subte_mensual)} meses')
    print(f'   Período: {subte_mensual.index[0].date()} -> {subte_mensual.index[-1].date()}')
    display(subte_mensual.head(8))

### 6.3 Peajes AUSA — Flujo Vehicular

In [ ]:
# ── Los archivos de AUSA pueden ser CSV o XLSX según el año ───────────────
# Completar con los links reales del portal una vez verificados
PEAJES_URLS = {
    # 2013: 'https://...',
    # ...
}

# ── Por ahora, cargar el archivo ya disponible localmente ─────────────────
local_files = list(DATA_DIR.glob('peajes_*.csv'))
print(f'Archivos locales de peajes encontrados: {[f.name for f in local_files]}')

if local_files:
    dfs = []
    for f in sorted(local_files):
        df = pd.read_csv(f, encoding='latin1', low_memory=False)
        df.columns = df.columns.str.lower().str.strip()
        dfs.append(df)
    peajes_raw = pd.concat(dfs, ignore_index=True)
    print(f'Registros totales cargados: {len(peajes_raw):,}')
    print('Columnas:', peajes_raw.columns.tolist())
else:
    print('⚠️  No hay archivos de peajes locales. Descargar manualmente del portal GCBA.')

In [ ]:
# ── Adaptar según las columnas reales del CSV ─────────────────────────────
# Ejecutar esta celda después de inspeccionar peajes_raw.columns

# peajes_raw['fecha'] = pd.to_datetime(peajes_raw['<col_fecha>'], dayfirst=True, errors='coerce')
# peajes_raw['cantidad'] = pd.to_numeric(peajes_raw['<col_cantidad>'], errors='coerce')
# peajes_mensual = peajes_raw.set_index('fecha')['cantidad'].resample('MS').sum()
# peajes_mensual.name = 'vehiculos'
# peajes_mensual.to_csv(DATA_DIR / 'peajes_mensual.csv')
# print(f'Peajes mensual: {len(peajes_mensual)} observaciones')

### 6.4 Dataset Consolidado y Variable Dummy COVID

In [ ]:
# ── Cargar series ya procesadas ───────────────────────────────────────────
subte   = pd.read_csv(DATA_DIR / 'subte_mensual.csv',  index_col=0, parse_dates=True).squeeze()
lluvia  = pd.read_csv(DATA_DIR / 'lluvia_mensual.csv', index_col=0, parse_dates=True).squeeze()
# peajes = pd.read_csv(DATA_DIR / 'peajes_mensual.csv', index_col=0, parse_dates=True).squeeze()

# ── Alinear al mismo período ──────────────────────────────────────────────
START, END = '2013-06-01', '2025-12-01'

df = pd.DataFrame({
    'pasajeros_subte': subte,
    # 'vehiculos_peajes': peajes,
    'lluvia_mm': lluvia,
}).loc[START:END]

# ── Dummy COVID: 1 para mar-2020 hasta dic-2021 ───────────────────────────
df['dummy_covid'] = 0
df.loc['2020-03-01':'2021-12-01', 'dummy_covid'] = 1

print(f'Dataset consolidado: {df.shape[0]} observaciones × {df.shape[1]} variables')
print(f'Meses COVID marcados: {df["dummy_covid"].sum()}')
df.head(10)

---
## 7. Análisis Exploratorio (EDA)

### 7.1 Series originales

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(df.index, df['pasajeros_subte'] / 1e6, color='steelblue', lw=1.5)
axes[0].set_title('Pasajeros de Subte (millones por mes)', fontweight='bold')
axes[0].set_ylabel('Millones')

# axes[1].plot(df.index, df['vehiculos_peajes'] / 1e6, color='darkorange', lw=1.5)
# axes[1].set_title('Flujo Vehicular en Peajes AUSA (millones por mes)', fontweight='bold')
# axes[1].set_ylabel('Millones')
axes[1].set_title('Flujo Vehicular en Peajes AUSA — pendiente de datos', color='gray')

axes[2].bar(df.index, df['lluvia_mm'], color='teal', alpha=0.7, width=20)
axes[2].set_title('Precipitaciones mensuales en CABA (mm)', fontweight='bold')
axes[2].set_ylabel('mm')

# Marcar período COVID
for ax in axes:
    ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-12-01'),
               alpha=0.15, color='red', label='COVID')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

axes[0].legend(['Pasajeros', 'Período COVID'], loc='upper left')
plt.tight_layout()
plt.savefig(DATA_DIR / 'series_originales.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.2 Descomposición STL

In [ ]:
def plot_stl(serie, titulo, periodo=12, color='steelblue'):
    """Descompone la serie con STL y grafica componentes."""
    stl = STL(serie.dropna(), period=periodo, robust=True)
    res = stl.fit()
    
    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    componentes = [
        (serie,           'Serie original', False),
        (res.trend,       'Tendencia',      False),
        (res.seasonal,    'Estacionalidad', False),
        (res.resid,       'Residuos',       True),
    ]
    for ax, (comp, nombre, es_resid) in zip(axes, componentes):
        if es_resid:
            ax.scatter(comp.index, comp, s=10, color=color, alpha=0.6)
            ax.axhline(0, color='black', lw=0.8, ls='--')
        else:
            ax.plot(comp.index, comp, color=color, lw=1.3)
        ax.set_ylabel(nombre, fontsize=10)
        ax.grid(True, alpha=0.3)
    axes[0].set_title(f'Descomposición STL — {titulo}', fontweight='bold')
    plt.tight_layout()
    plt.show()
    return res

stl_subte = plot_stl(df['pasajeros_subte'], 'Pasajeros de Subte', color='steelblue')

In [ ]:
stl_lluvia = plot_stl(df['lluvia_mm'], 'Precipitaciones CABA', color='teal')

---
## 8. Estacionariedad

### 8.1 Funciones de autocorrelación

In [ ]:
def plot_acf_pacf(serie, titulo, lags=36, color='steelblue'):
    """Grafica FAC y FACP de la serie."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    plot_acf(serie.dropna(),  lags=lags, ax=axes[0], color=color, title=f'FAC — {titulo}')
    plot_pacf(serie.dropna(), lags=lags, ax=axes[1], color=color, title=f'FACP — {titulo}')
    plt.tight_layout()
    plt.show()

plot_acf_pacf(df['pasajeros_subte'], 'Subte (serie original)')

In [ ]:
plot_acf_pacf(df['lluvia_mm'], 'Precipitaciones (serie original)', color='teal')

### 8.2 Tests de Raíces Unitarias (ADF + KPSS)

In [ ]:
def test_estacionariedad(serie, nombre):
    """Corre ADF y KPSS e imprime la interpretación conjunta."""
    s = serie.dropna()
    
    # ADF — H0: tiene raíz unitaria (no estacionaria)
    adf_stat, adf_p, _, _, adf_cv, _ = adfuller(s, autolag='AIC', regression='ct')
    
    # KPSS — H0: es estacionaria (en tendencia)
    kpss_stat, kpss_p, _, kpss_cv = kpss(s, regression='ct', nlags='auto')
    
    print(f'{'═'*55}')
    print(f'  Serie: {nombre}')
    print(f'{'─'*55}')
    print(f'  ADF  — estadístico: {adf_stat:.4f}   p-value: {adf_p:.4f}')
    print(f'  KPSS — estadístico: {kpss_stat:.4f}   p-value: {kpss_p:.4f}')
    print(f'{'─'*55}')
    
    rechaza_adf  = adf_p  < 0.05   # rechaza H0 → evidencia de estacionariedad
    rechaza_kpss = kpss_p < 0.05   # rechaza H0 → evidencia de NO estacionariedad
    
    if rechaza_adf and not rechaza_kpss:
        print('  → Ambos coinciden: la serie ES estacionaria.')
    elif not rechaza_adf and rechaza_kpss:
        print('  → Ambos coinciden: la serie NO ES estacionaria.')
    elif not rechaza_adf and not rechaza_kpss:
        print('  → Estacionaria por tendencia (eliminar tendencia para estacionariedad estricta).')
    else:
        print('  → Estacionaria por diferencia (aplicar diferenciación).')
    print()

test_estacionariedad(df['pasajeros_subte'], 'Subte')
test_estacionariedad(df['lluvia_mm'],       'Precipitaciones')

### 8.3 Diferenciación (si corresponde)

In [ ]:
# ── Diferenciación regular (d=1) ──────────────────────────────────────────
subte_diff1 = df['pasajeros_subte'].diff(1).dropna()

# ── Diferenciación estacional (D=1, s=12) ─────────────────────────────────
subte_diff1_12 = subte_diff1.diff(12).dropna()

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
axes[0].plot(df['pasajeros_subte'],  color='steelblue', lw=1.3, label='Original')
axes[1].plot(subte_diff1,            color='darkorange',lw=1.3, label='Δ¹')
axes[2].plot(subte_diff1_12,         color='green',     lw=1.3, label='Δ¹Δ¹²')
for ax in axes:
    ax.axhline(0, color='black', lw=0.7, ls='--')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
axes[0].set_title('Subte — Original y diferenciaciones', fontweight='bold')
plt.tight_layout()
plt.show()

test_estacionariedad(subte_diff1,    'Subte — diff(1)')
test_estacionariedad(subte_diff1_12, 'Subte — diff(1) diff(12)')

In [ ]:
# FAC/FACP de la serie diferenciada
plot_acf_pacf(subte_diff1_12, 'Subte — diff(1) diff(12)')

---
## 9. Modelado SARIMA

### 9.1 Train / Test Split

In [ ]:
TRAIN_END = '2023-12-01'
TEST_START = '2024-01-01'

train = df.loc[:TRAIN_END]
test  = df.loc[TEST_START:]

print(f'Train: {len(train)} obs  ({train.index[0].date()} → {train.index[-1].date()})')
print(f'Test:  {len(test)}  obs  ({test.index[0].date()} → {test.index[-1].date()})')

### 9.2 Grilla de parámetros SARIMA

In [ ]:
from itertools import product

def grilla_sarima(serie, p_rng, d_rng, q_rng, P_rng, D_rng, Q_rng, s=12, verbose=False):
    """Itera combinaciones SARIMA y devuelve resultados ordenados por AIC."""
    resultados = []
    combos = list(product(p_rng, d_rng, q_rng, P_rng, D_rng, Q_rng))
    print(f'Evaluando {len(combos)} modelos...')
    
    for i, (p, d, q, P, D, Q) in enumerate(combos):
        try:
            model = SARIMAX(
                serie,
                order=(p, d, q),
                seasonal_order=(P, D, Q, s),
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit(disp=False)
            resultados.append({
                'orden': f'SARIMA({p},{d},{q})({P},{D},{Q})[{s}]',
                'AIC': round(model.aic, 2),
                'BIC': round(model.bic, 2),
                'model': model,
            })
        except Exception:
            pass
        if verbose and (i + 1) % 20 == 0:
            print(f'  {i+1}/{len(combos)} completados')
    
    res_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'model'} for r in resultados])
    res_df = res_df.sort_values('AIC').reset_index(drop=True)
    modelos = {r['orden']: r['model'] for r in resultados}
    return res_df, modelos

# ── Correr grilla para Subte ──────────────────────────────────────────────
resultados_subte, modelos_subte = grilla_sarima(
    train['pasajeros_subte'],
    p_rng=range(0, 3), d_rng=[1], q_rng=range(0, 3),
    P_rng=range(0, 2), D_rng=[1], Q_rng=range(0, 2),
    s=12, verbose=True
)

print('\nTop 10 modelos por AIC:')
resultados_subte.head(10)

### 9.3 Selección del modelo y diagnóstico de residuos

In [ ]:
# ── Seleccionar el mejor modelo ───────────────────────────────────────────
mejor_orden = resultados_subte.iloc[0]['orden']
modelo_subte = modelos_subte[mejor_orden]
print(f'Modelo seleccionado: {mejor_orden}')
print(modelo_subte.summary())

In [ ]:
modelo_subte.plot_diagnostics(figsize=(14, 8))
plt.suptitle(f'Diagnóstico de residuos — {mejor_orden}', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
residuos = modelo_subte.resid.dropna()

lb = acorr_ljungbox(residuos, lags=[10, 20], return_df=True)
jb_stat, jb_p = jarque_bera(residuos)

print('Ljung-Box (lags 10 y 20):')
print(lb)
print(f'\nJarque-Bera: estadístico={jb_stat:.4f}, p-value={jb_p:.4f}')
print('→ Residuos normales' if jb_p > 0.05 else '→ Residuos NO normales (ver Q-Q)')

### 9.4 Métricas sobre el conjunto de test

In [ ]:
def evaluar_modelo(modelo, test_serie, titulo='', color='steelblue'):
    """Genera predicciones para el período de test y calcula métricas."""
    pred = modelo.get_forecast(steps=len(test_serie))
    pred_mean = pred.predicted_mean
    pred_ci   = pred.conf_int(alpha=0.05)
    pred_mean.index = test_serie.index
    pred_ci.index   = test_serie.index

    rmse = np.sqrt(mean_squared_error(test_serie, pred_mean))
    mae  = mean_absolute_error(test_serie, pred_mean)
    mape = mean_absolute_percentage_error(test_serie, pred_mean) * 100

    print(f'{titulo}')
    print(f'  RMSE: {rmse:,.0f}   |   MAE: {mae:,.0f}   |   MAPE: {mape:.2f}%')

    fig, ax = plt.subplots(figsize=(14, 4))
    train_serie = modelo.model.endog
    train_idx   = modelo.model.data.dates
    ax.plot(train_idx, train_serie, color='gray', lw=1, label='Train', alpha=0.7)
    ax.plot(test_serie.index, test_serie, color='steelblue', lw=1.5, label='Test real')
    ax.plot(pred_mean.index, pred_mean, color=color, lw=1.5, ls='--', label='Predicción')
    ax.fill_between(pred_ci.index, pred_ci.iloc[:, 0], pred_ci.iloc[:, 1],
                    alpha=0.2, color=color, label='IC 95%')
    ax.set_title(f'Pronóstico vs. Real — {titulo}', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()

    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

metricas_subte = evaluar_modelo(modelo_subte, test['pasajeros_subte'], titulo=mejor_orden)

### 9.5 Pronóstico

In [ ]:
# ── Re-entrenar sobre toda la serie y pronosticar 12 meses ────────────────
# (completar con los parámetros del modelo seleccionado)

# modelo_final = SARIMAX(df['pasajeros_subte'], order=(...), seasonal_order=(...)).fit(disp=False)
# forecast = modelo_final.get_forecast(steps=12)
# ...

---
## 10. Modelo SARIMA-X: Subte con Precipitaciones

### 10.1 Justificación

Se incorpora la precipitación mensual como regresor exógeno en el modelo SARIMA para el subte, junto con la variable dummy de COVID-19. La hipótesis (H6) es que ante mayor lluvia, la demanda de subte aumenta porque los usuarios sustituyen modos de transporte activo (caminata, bicicleta) o evitan el auto. Se espera que el coeficiente de `lluvia_mm` sea **positivo y estadísticamente significativo**.

In [ ]:
# ── Regresores exógenos ───────────────────────────────────────────────────
exog_train = train[['lluvia_mm', 'dummy_covid']]
exog_test  = test[['lluvia_mm', 'dummy_covid']]

# ── Ajustar SARIMA-X con los parámetros del mejor modelo puro ─────────────
# (completar order y seasonal_order con los valores seleccionados en sección 9)

# modelo_sarimax = SARIMAX(
#     train['pasajeros_subte'],
#     exog=exog_train,
#     order=(...),
#     seasonal_order=(..., 12),
#     enforce_stationarity=False,
#     enforce_invertibility=False,
# ).fit(disp=False)

# print(modelo_sarimax.summary())

In [ ]:
# ── Comparación SARIMA puro vs. SARIMA-X ──────────────────────────────────

# comparacion = pd.DataFrame({
#     'Modelo': ['SARIMA puro', 'SARIMA-X'],
#     'AIC': [modelo_subte.aic, modelo_sarimax.aic],
#     'BIC': [modelo_subte.bic, modelo_sarimax.bic],
#     'MAPE (%)': [metricas_subte['MAPE'], metricas_sarimax['MAPE']],
# })
# comparacion

---
## 11. Conclusiones

> *Esta sección se completa al finalizar el análisis.*

- [ ] ¿Las tres series son no estacionarias? ¿Requieren d=1, D=1?
- [ ] ¿El subte y los peajes tienen estacionalidad opuesta?
- [ ] ¿El coeficiente de lluvia en el SARIMA-X es significativo y con el signo esperado?
- [ ] ¿El SARIMA-X supera al SARIMA puro en AIC/BIC y métricas de test?
- [ ] Limitaciones: quiebre COVID, expansión de la red de subte, datos faltantes en peajes.